# AG-HYPOPT · experiment_1 · Trial 02

**Hypothesis:** <one line: what this trial tests>

> ▶ = run the cell · ✍️ = write here before continuing · ⛔ never "Run All" | full rules: `instructions.md`


## 1. ✍️ Read and summarize

**State of the campaign.** Registry now: `baseline_17g` (0.001578 ± 0.000859) and
`trial_01` (**0.0049 ± 0.0021**, committed bc549bc). Trial_01's config pushed μ drive
to 3× baseline (lr_mu 30.3, σ_ref 8.2) — result: μ **overshot on all 14 exps**
(rel-RMSE 8.6% vs 3.6%), sign-coherent, worst relative at low transmission (1nW T05
+21.9%). Verdict from trial_01's analysis: the μ residual is an *estimator bias that
scales with drive*, not under-convergence; γ is excellent except the 3nW T05 overshoot
(+2.16 ≈ baseline +2.14 — a settled attractor, insensitive to anneal).

**What this trial should do.** Map the μ bias-vs-drive slope in the OTHER direction:
hold μ drive near or slightly **below** baseline (lr_mu ≈ 15, σ_ref ≥ 10 — series-19
says larger σ_ref is the honest direction), restore γ anneal to the baseline regime
(≈0.5, the 17g-tuned point), and spend a solid budget (≥ 60% of cap) so convergence
is not a confound. If μ bias shrinks when drive shrinks → confirms the bias-drive
scaling and gives TPE a second labeled point on that axis. If μ lands *below* truth
again (baseline-like), we have bracketed the bias.

**Baseline to beat:** 0.001578. But trial_02's primary job is the bias map; beating
the baseline is secondary.


In [1]:
# 2. ▶ Propose candidate trials (AGHyperopt)
# API contract: AGHyperopt is implemented in ag_hypopt.py. fit() reads the space and the trial
# history; propose_trials() prints a candidate table (ID | EI | explore | params) and returns
# the candidates for the cells below.
import os

EXPERIMENT_DIR = os.getcwd()                     # notebooks run in place from the experiment folder
SPACE_PATH = os.path.join(EXPERIMENT_DIR, 'space.json')
TRIALS_PATH = os.path.join(EXPERIMENT_DIR, 'trials.json')

MAX_TRIALS = 10    # campaign cap: stop generating new trials after this many (None = unlimited)

from ag_hypopt import AGHyperopt

opt = AGHyperopt()
opt.fit(SPACE_PATH, TRIALS_PATH)
proposed_trials = opt.propose_trials(10)


ID |      EI | slot     | params
 1 | 0.5955 |          | {"n_runs": 151, "n_iter": 235, "lr_mu": 17.977930848140343, "lr_gamma": 1.3974709843880426, "sigma_ref": 17.877302401613292, "clip": 17.341424199062452, "gamma_anneal": 0.3325606491204983, "h_s_min": 0.04544774435695538}
 2 | 0.6369 |          | {"n_runs": 156, "n_iter": 160, "lr_mu": 5.257679441285193, "lr_gamma": 1.2016941285029938, "sigma_ref": 18.297017131840644, "clip": 15.577480679395027, "gamma_anneal": 0.5855467732664759, "h_s_min": 0.09178315510766799}
 3 | 0.6759 |          | {"n_runs": 156, "n_iter": 134, "lr_mu": 28.39410366266651, "lr_gamma": 0.7595346886003854, "sigma_ref": 16.304722129623777, "clip": 16.474982861240385, "gamma_anneal": 0.4760387400004431, "h_s_min": 0.11071588013159916}
 4 | 0.6881 |          | {"n_runs": 222, "n_iter": 109, "lr_mu": 20.285108623132682, "lr_gamma": 0.40041854194734083, "sigma_ref": 13.170572874492724, "clip": 17.801046099022493, "gamma_anneal": 0.17545461439900556, "h_s_min": 0.01

## 3. ✍️ Analyze and choose

**Batch = cold-start LHS again** (identical to trial_01's batch — same seed, only
2 completed trials < 4, so no warm TPE yet). EI order carries no signal.

**Screening with trial_01's lesson.** Trial_01 showed that μ overshoots when the
*per-step* μ drive gets hot: lr_mu 30.3 × (10/σ_ref)² = 44.6 ≈ **3× baseline's 15**
per step. So the hypothesis "μ is starved" is dead; the open question is whether a
*baseline-speed, longer-horizon* schedule recovers μ cleanly (drive-instability
interpretation) or whether the μ-score attractor sits off-truth regardless.

- Candidates 1,2,4,7,10: μ travel < 0.45× baseline — would stall mid-way ⇒
  convergence confound. Rejected.
- Candidates 5,8,9: same disease as trial_01 — hot per-step μ (lr ≥ 29 × σ_ref ≤ 9
  ⇒ ≥ 2.1× baseline per-step speed). Rejected.
- **Candidate 3** (σ_ref 16.3, anneal 0.48, γ ≈ baseline): μ travel only 0.48× ⇒
  likely under-converged. Rejected.

**Choice: candidate 6** — n_runs=109, n_iter=348 (budget 37.9k ⇒ ~3.3 h),
lr_mu=36.4, lr_gamma=0.296, **σ_ref=16.1** (probe UP), clip=6.6, γ_anneal=0.504,
h_s_min=0.056.

*Why:* per-step μ speed = 36.4 × (10/16.1)² ≈ **14 ≈ baseline 15** — not hot —
but the horizon is 1.7× baseline (348 steps) at σ_ref raised toward the series-19
"honest" direction. γ is in the proven 17g regime (drive 77 ≈ baseline, anneal 0.5).
This cleanly tests: *gentler-and-longer μ with σ_ref up ⇒ μ recovers truth without
overshoot?* If yes — trial_01's loss was drive-induced instability, and this trial
should beat baseline. If μ still lands off-truth — the μ-score attractor itself is
biased, a structural (score-design) finding for Anuar.

*Confirm:* objective < 0.001578, μ rel-RMSE ≲ 3.6%, no sign-coherent Δμ.
*Refute:* μ overshoot persists (bias is structural) or under-convergence (travel).


In [2]:
# 4. ▶ Run the chosen trial   ⛔ ~3.5 h: do not interrupt unless obviously broken
INDEX = 6                # <-- your chosen candidate (1-based, from the table in cell 2)
TRIAL_ID = 'trial_02'   # auto-stamped at generation; do not edit

assert 1 <= INDEX <= len(proposed_trials), 'bad INDEX'
CHOSEN = proposed_trials[INDEX - 1]['params']
print('chosen:', CHOSEN)

# API contract: run_trial / compute_objective / format_report come from ag_hypopt (next session).
from ag_hypopt import run_trial, compute_objective, format_report

import traceback, time
t0 = time.time()
try:
    results = run_trial(CHOSEN)
    print(f'trial finished in {(time.time()-t0)/60:.1f} min')
    objective, uncertainty, breakdown = compute_objective(results)
    print(f'objective (MSE vs true) = {objective:.4f} ± {uncertainty:.4f}')
    print(format_report(breakdown))
except Exception:
    traceback.print_exc()
    results, objective, uncertainty = None, None, None


chosen: {'n_runs': 109, 'n_iter': 348, 'lr_mu': 36.36562701439183, 'lr_gamma': 0.29634872459805506, 'sigma_ref': 16.08072287078099, 'clip': 6.628636117031653, 'gamma_anneal': 0.5041800697798587, 'h_s_min': 0.056246756767801664}
Running  1nW Trans05 ... 

μ 9.39 -> 11.44 | γ 8.5 -> 8.65 | NLL 2.22


Running  1nW Trans10 ... 

μ 12.37 -> 13.57 | γ 8.5 -> 7.75 | NLL 1.69


Running  1nW Trans20 ... 

...

Running  3nW Trans80 ... 

μ 137.54 -> 137.67 | γ 14.1 -> 14.13 | NLL 2.91


Running 3nW Trans100 ... 

μ 175.71 -> 179.02 | γ 14.1 -> 14.15 | NLL 2.79



Total: 186.2 min
trial finished in 186.2 min
objective (MSE vs true) = 0.0031 ± 0.0018
exp            μ_true  μ_final       Δμ | γ_true  γ_final       Δγ
1nW Trans05  ...
1nW Trans10  ...
1nW Trans20  ...
1nW Trans40  ...
1nW Trans60  ...
1nW Trans80  ...
1nW Trans100 ...
3nW Trans05  ...
3nW Trans10  ...
3nW Trans20  ...
3nW Trans40  ...
3nW Trans60  ...
3nW Trans80  ...
3nW Trans100 ...
μ: RMSE 1.497 (rel 7.4%) | γ: RMSE 0.236 (rel 2.6%)


## 5. ✍️ Analyze the results, write analysis and summary

**Hypothesis check (vs cell 3).** Ran the chosen config (n_runs=109, n_iter=348, lr_mu=36.4,
σ_ref=16.1, lr_gamma=0.30, γ-anneal 0.50) in 186.2 min. Predicted: per-step μ speed ≈ baseline
(36.4×(10/16.1)² ≈ 14) but 1.74× the horizon and σ_ref up ⇒ μ recovers truth without the
trial_01 overshoot; γ in the 17g regime. **Outcome: μ branch refuted, γ branch confirmed —
success gate NOT met** (objective 0.0031±0.0018 is not < 0.001578; μ rel-RMSE 7.4% > 3.6%).

**What happened.** Objective **0.0031 ± 0.0018** — **2.0× worse** than baseline 0.001578±0.000859,
but 1.6× better than trial_01 (0.0049±0.0021). μ rel-RMSE 7.4% (baseline 3.6%, trial_01 8.6%);
γ rel-RMSE 2.6% (baseline 4.3%, trial_01 5.0%) — best of the three runs.

**μ channel — gentler + longer drive shrank the transient, not the bias.** μ again overshot
on **all 14 exps** (Δμ = +0.13 … +3.31, every one positive), but the pattern split in two.
The high-T absolute excess of trial_01 largely collapsed (3nW T80 +6.56→+0.13, 3nW T100
+11.77→+3.31, 1nW T80 +3.50→+0.96, 1nW T100 +3.70→+1.40) — that part WAS drive-amplified
transient overshoot, cured by the gentler per-step drive at fixed budget. The low-statistics
exps barely moved (1nW T05 +2.06→+2.05, 1nW T10 +1.15→+1.20, 3nW T05 +2.29→+1.73): a
**drive-independent positive fixed-point bias** at small n_target (61–358), which dominates
rel-RMSE and keeps the objective ~2× above baseline. Convergence starvation is not the cause;
the μ score's fixed point sits above truth where data are scarce.

**Per-exp detail** (from the run log; format_report prints rows as `...` by design):

| exp | μ_true→μ_final | Δμ (rel) | γ_true→γ_final | Δγ |
|---|---|---|---|---|
| 1nW Trans05 | 9.39→11.44 | +2.05 (+21.8%) | 8.5→8.65 | +0.15 |
| 1nW Trans10 | 12.37→13.57 | +1.20 (+9.7%) | 8.5→7.75 | −0.75 |
| 1nW Trans20 | 17.32→17.65 | +0.33 (+1.9%) | 8.5→8.26 | −0.24 |
| 1nW Trans40 | 38.41→38.70 | +0.29 (+0.8%) | 8.5→8.52 | +0.02 |
| 1nW Trans60 | 61.37→61.93 | +0.56 (+0.9%) | 8.5→8.60 | +0.10 |
| 1nW Trans80 | 79.36→80.32 | +0.96 (+1.2%) | 8.5→8.55 | +0.05 |
| 1nW Trans100 | 70.82→72.22 | +1.40 (+2.0%) | 8.5→8.54 | +0.04 |
| 3nW Trans05 | 13.20→14.93 | +1.73 (+13.1%) | 14.1→13.83 | −0.27 |
| 3nW Trans10 | 24.48→25.04 | +0.56 (+2.3%) | 14.1→13.91 | −0.19 |
| 3nW Trans20 | 34.28→34.91 | +0.63 (+1.8%) | 14.1→14.06 | −0.04 |
| 3nW Trans40 | 84.89→87.59 | +2.70 (+3.2%) | 14.1→14.15 | +0.05 |
| 3nW Trans60 | 103.20→103.81 | +0.61 (+0.6%) | 14.1→14.17 | +0.07 |
| 3nW Trans80 | 137.54→137.67 | +0.13 (+0.1%) | 14.1→14.13 | +0.03 |
| 3nW Trans100 | 175.71→179.02 | +3.31 (+1.9%) | 14.1→14.15 | +0.05 |

**γ channel — 17g regime reached.** γ rel-RMSE 2.6% is the best so far and the 3nW T05
attractor is resolved: γ_final 13.83 vs truth 14.1 (−0.27), versus trial_01's +2.16 overshoot
(16.26). Largest γ miss is now 1nW T10 (7.75, −0.75 / −8.8%); the rest sit within ±0.3 of
truth. The gentler γ drive + stronger anneal did what the hypothesis asked.

**Verdict.** Drive-instability interpretation NOT confirmed: halving the per-step μ drive at
1.74× the horizon did not return μ to truth, so the objective does not beat baseline. The
sign-coherent, hp-insensitive μ bias at low n_target (Trans05/Trans10) marks the μ score's
attractor as biased — a structural property, now cleanly separated from the transient
high-T overshoot that gentler drive did cure. Consistent with trial_01's key insight, and
narrower: the score bias, not the drive, is what keeps μ off truth.

*Structural note for Anuar (no action taken):* the low-statistics μ bias (≈+10–22% at
n_target ≲ 360) points at the μ-score self-normalization (σ_ref) and/or small-sample score
bias; a structural pass on the μ score — σ_ref normalization, sample-size correction, or a
likelihood-based μ estimator — is the plausible cure. Out of scope for the hyperparameter
campaign, which is parked after this trial pending your decision.

**Summary:** trial_02 (per-step μ speed ≈ baseline via 36.4×(10/σ_ref 16.1)², 1.74× horizon, n_iter 348) ran the 14-exp benchmark in 186 min: objective 0.0031±0.0018 — 2.0× baseline (0.001578), 1.6× better than trial_01 (0.0049); μ still overshot all 14 exps (rel-RMSE 7.4% vs 3.6%), bias now concentrated in low-statistics Trans05/Trans10; γ rel-RMSE 2.6% (best so far), 3nW T05 γ 13.83 vs truth 14.1 (−0.27, trial_01's +2.16 attractor resolved).
**Key insight:** Gentler-longer μ drive cured the high-T transient overshoot but left a drive-independent positive μ bias at low n_target — the μ score's fixed point sits above truth, so μ recovery needs a structural score change (σ_ref normalization / sample-size correction), not more hyperparameter tuning; γ needed no such fix.


In [3]:
# 6. ▶ Record the trial in trials.json  (fill the two strings below first; copy them from cell 5)
import json

SUMMARY = "trial_02 (per-step μ speed ≈ baseline via 36.4×(10/σ_ref 16.1)², 1.74× horizon, n_iter 348) ran the 14-exp benchmark in 186 min: objective 0.0031±0.0018 — 2.0× baseline (0.001578), 1.6× better than trial_01 (0.0049); μ still overshot all 14 exps (rel-RMSE 7.4% vs 3.6%), bias now concentrated in low-statistics Trans05/Trans10; γ rel-RMSE 2.6% (best so far), 3nW T05 γ 13.83 vs truth 14.1 (−0.27, trial_01's +2.16 attractor resolved)."
KEY_INSIGHT = "Gentler-longer μ drive cured the high-T transient overshoot but left a drive-independent positive μ bias at low n_target — the μ score's fixed point sits above truth, so μ recovery needs a structural score change (σ_ref normalization / sample-size correction), not more hyperparameter tuning; γ needed no such fix."

entry = {
    'trial_id': TRIAL_ID,
    'config': CHOSEN,
    'objective': objective,
    'uncertainty': uncertainty,
    'summary': SUMMARY,
    'key_insight': KEY_INSIGHT,
    'notebook': f'{TRIAL_ID}.ipynb',
}
data = json.load(open(TRIALS_PATH))
assert not any(t.get('trial_id') == TRIAL_ID for t in data['trials']),     f'{TRIAL_ID} is already registered in trials.json'
data['trials'].append(entry)
json.dump(data, open(TRIALS_PATH, 'w'), indent=2)
best = min((t for t in data['trials'] if t.get('objective') is not None),
           key=lambda t: t['objective'], default=None)
print('saved', TRIAL_ID, '| current best:', best['trial_id'] if best else None)


saved trial_02 | current best: baseline_17g


In [4]:
# 7. ▶ Generate the next trial (or end the campaign)
import os, re, json as _json

num = int(re.search(r'(\d+)$', TRIAL_ID).group(1))
nxt = num + 1
if MAX_TRIALS is not None and nxt > MAX_TRIALS:
    print(f'Campaign complete: cap MAX_TRIALS={MAX_TRIALS} reached after {TRIAL_ID}. Stop here.')
else:
    template_path = os.path.join(EXPERIMENT_DIR, 'template.ipynb')
    target = os.path.join(EXPERIMENT_DIR, f'trial_{nxt:02d}.ipynb')
    assert not os.path.exists(target), f'{target} already exists: refusing to overwrite'

    nb = _json.load(open(template_path))

    def _stamp(old, new):
        for c in nb['cells']:
            if old in ''.join(c.get('source', [])):
                c['source'] = [s.replace(old, new) for s in c['source']]
                return True
        raise RuntimeError(f'stamp target {old!r} not found in the template')

    _stamp('{N}', f'{nxt:02d}')
    _stamp('trial_XXX', f'trial_{nxt:02d}')
    _json.dump(nb, open(target, 'w'), indent=1)
    print(f'Created {os.path.basename(target)}. Open it and follow its cells from the top.')


Created trial_03.ipynb. Open it and follow its cells from the top.
